# 04 BERT Weighted-Loss Baseline (LIAR)

**Goal:** test whether adding class weighting to the BERT training loss improves the LIAR binary baseline, especially recall on the `FAKE` class.

This notebook keeps the same structure as `03_bert_baseline.ipynb` and changes only one core training detail:
- compute class weights from the **train split only**
- apply those weights in the training loss

Everything else stays aligned with the current BERT setup:
- same LIAR train / valid / test splits
- same binary label mapping
- same input field: `statement`
- same model: `bert-base-uncased`
- same validation macro-F1 checkpoint selection
- same final reporting style


## Why this experiment is separate

The original BERT notebook is kept unchanged.

This notebook is a clean follow-up experiment that reuses the same project utilities and evaluation style, while isolating the class-weighted loss idea for fair comparison.


In [ ]:
# Optional on a fresh environment:
# !pip -q install transformers torch scikit-learn pandas

import copy
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

from liar_utils import (
    ID2LABEL,
    dataframe_to_markdown,
    evaluate_predictions,
    load_binary_dataset_splits,
    make_liar_config,
    show_binary_distribution,
)

DATA_DIR = Path("liar_dataset")
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
RANDOM_SEED = 42

# Unweighted BERT seed-42 reference from the successful run.
BERT_BASELINE_VALID_ACC = 0.6386
BERT_BASELINE_VALID_MACRO_F1 = 0.6309
BERT_BASELINE_TEST_ACC = 0.6401
BERT_BASELINE_TEST_MACRO_F1 = 0.6231
BERT_BASELINE_FAKE_RECALL = 0.4901

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(RANDOM_SEED)

print("Parameters:")
print("  DATA_DIR =", DATA_DIR)
print("  MODEL_NAME =", MODEL_NAME)
print("  MAX_LENGTH =", MAX_LENGTH)
print("  TRAIN_BATCH_SIZE =", TRAIN_BATCH_SIZE)
print("  EVAL_BATCH_SIZE =", EVAL_BATCH_SIZE)
print("  NUM_EPOCHS =", NUM_EPOCHS)
print("  LEARNING_RATE =", LEARNING_RATE)
print("  WEIGHT_DECAY =", WEIGHT_DECAY)
print("  RANDOM_SEED =", RANDOM_SEED)
print("  DEVICE =", DEVICE)
print("  Fixed label mapping: REAL=0, FAKE=1")


## 1) Load LIAR and apply the same binary setup as the unweighted BERT baseline


In [ ]:
dataset_config = make_liar_config(DATA_DIR)

train_p, valid_p, test_p = load_binary_dataset_splits(dataset_config)

print("Prepared shapes:")
print("  train_p:", train_p.shape)
print("  valid_p:", valid_p.shape)
print("  test_p :", test_p.shape)

show_binary_distribution(train_p, "Train")
show_binary_distribution(valid_p, "Valid")
show_binary_distribution(test_p, "Test")


## 2) Compute class weights from the train split only

The weighted loss uses only the class distribution of the training split. Validation and test distributions are not used in this calculation.


In [ ]:
train_counts = train_p["y"].value_counts().sort_index()
num_classes = len(ID2LABEL)
num_train = len(train_p)

# Balanced weights: n_samples / (n_classes * class_count)
class_weights = torch.tensor(
    [
        num_train / (num_classes * train_counts.loc[class_id])
        for class_id in sorted(ID2LABEL)
    ],
    dtype=torch.float,
    device=DEVICE,
)

print("Train split counts:")
print(train_counts)
print("\nClass weights used in training loss:")
for class_id, class_name in ID2LABEL.items():
    print(f"  {class_name} ({class_id}): {class_weights[class_id].item():.4f}")


## 3) Tokenization and dataset helpers


In [ ]:
class EncodedTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length: int):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict:
        item = {key: value[idx] for key, value in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


def make_dataloader(
    df: pd.DataFrame,
    tokenizer,
    text_column: str,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = EncodedTextDataset(
        texts=df[text_column].tolist(),
        labels=df["y"].tolist(),
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


## 4) Training and evaluation helpers

Only the training loss is changed here. Validation and test metrics are still computed the same way as in the unweighted BERT baseline.


In [ ]:
def move_batch_to_device(batch: dict, device: torch.device) -> dict:
    return {key: value.to(device) for key, value in batch.items()}


def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    device: torch.device,
    class_weights: torch.Tensor,
) -> dict:
    model.train()
    total_loss = 0.0
    all_labels = []
    all_preds = []

    for batch in dataloader:
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad()

        labels = batch["labels"]
        model_inputs = {key: value for key, value in batch.items() if key != "labels"}
        outputs = model(**model_inputs)
        loss = F.cross_entropy(outputs.logits, labels, weight=class_weights)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        preds = outputs.logits.detach().cpu().argmax(dim=1)
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.detach().cpu().tolist())

    metrics = evaluate_predictions(all_labels, all_preds)
    metrics["loss"] = total_loss / max(len(dataloader), 1)
    return metrics


def evaluate_model(model, dataloader, device: torch.device) -> dict:
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []

    with torch.inference_mode():
        for batch in dataloader:
            batch = move_batch_to_device(batch, device)
            outputs = model(**batch)
            total_loss += outputs.loss.item()

            preds = outputs.logits.detach().cpu().argmax(dim=1)
            labels = batch["labels"].detach().cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    metrics = evaluate_predictions(all_labels, all_preds)
    metrics["loss"] = total_loss / max(len(dataloader), 1)
    metrics["preds"] = all_preds
    return metrics


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id={label_name: label_id for label_id, label_name in ID2LABEL.items()},
)
model.to(DEVICE)

train_loader = make_dataloader(
    train_p,
    tokenizer,
    text_column=dataset_config.text_column,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
)
valid_loader = make_dataloader(
    valid_p,
    tokenizer,
    text_column=dataset_config.text_column,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)
test_loader = make_dataloader(
    test_p,
    tokenizer,
    text_column=dataset_config.text_column,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

total_training_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print("Tokenizer and model are ready.")
print("  train batches:", len(train_loader))
print("  valid batches:", len(valid_loader))
print("  test batches :", len(test_loader))
print("  warmup steps :", warmup_steps)


In [ ]:
history_rows = []
best_state_dict = None
best_epoch = None
best_valid_score = (-1.0, -1.0)

for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        DEVICE,
        class_weights,
    )
    valid_metrics = evaluate_model(model, valid_loader, DEVICE)

    history_rows.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "valid_loss": valid_metrics["loss"],
            "valid_accuracy": valid_metrics["accuracy"],
            "valid_macro_f1": valid_metrics["macro_f1"],
        }
    )

    current_valid_score = (valid_metrics["macro_f1"], valid_metrics["accuracy"])
    if current_valid_score > best_valid_score:
        best_valid_score = current_valid_score
        best_epoch = epoch
        best_state_dict = copy.deepcopy(model.state_dict())
        checkpoint_note = " <- saved best checkpoint"
    else:
        checkpoint_note = ""

    print(f"Epoch {epoch}/{NUM_EPOCHS}{checkpoint_note}")
    print(
        f"  Train: loss={train_metrics['loss']:.4f}, "
        f"acc={train_metrics['accuracy']:.4f}, "
        f"macro_f1={train_metrics['macro_f1']:.4f}"
    )
    print(
        f"  Valid: loss={valid_metrics['loss']:.4f}, "
        f"acc={valid_metrics['accuracy']:.4f}, "
        f"macro_f1={valid_metrics['macro_f1']:.4f}"
    )

history_df = pd.DataFrame(history_rows)
history_df


## 5) Load the best validation checkpoint and evaluate on the test split

The best checkpoint is selected by validation macro-F1 and must be loaded back into the model before the final test evaluation.


In [ ]:
if best_state_dict is None:
    raise RuntimeError("Training did not produce a checkpoint.")

model.load_state_dict(best_state_dict)
print(f"Loaded best_state_dict into model from epoch {best_epoch} before test evaluation.")

best_valid_metrics = evaluate_model(model, valid_loader, DEVICE)
test_metrics = evaluate_model(model, test_loader, DEVICE)

print("Best validation checkpoint:")
print(f"  best_epoch: {best_epoch}")
print(f"  valid accuracy: {best_valid_metrics['accuracy']:.4f}")
print(f"  valid macro-F1: {best_valid_metrics['macro_f1']:.4f}")

print("\nFinal weighted BERT test result:")
print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Macro-F1: {test_metrics['macro_f1']:.4f}")

print("\nConfusion matrix (label order: [0, 1] = [REAL, FAKE]):")
print(test_metrics["confusion_matrix"])

print("\nClassification report:")
print(test_metrics["classification_report"])


## 6) Optional: save a markdown run summary


In [ ]:
run_date = datetime.now().strftime("%Y-%m-%d %H:%M")

history_export = history_df.copy()
for column in [
    "train_loss",
    "train_accuracy",
    "train_macro_f1",
    "valid_loss",
    "valid_accuracy",
    "valid_macro_f1",
]:
    history_export[column] = history_export[column].map(lambda value: f"{value:.4f}")

summary_lines = []
summary_lines.append("# LIAR BERT Weighted-Loss Run Output")
summary_lines.append("")
summary_lines.append(f"- Date: {run_date}")
summary_lines.append("- Task: Binary classification")
summary_lines.append("- Input field: statement")
summary_lines.append("- Label mapping: REAL=0, FAKE=1")
summary_lines.append(f"- Model: {MODEL_NAME}")
summary_lines.append(f"- Max length: {MAX_LENGTH}")
summary_lines.append(f"- Epochs: {NUM_EPOCHS}")
summary_lines.append(f"- Learning rate: {LEARNING_RATE}")
summary_lines.append("")
summary_lines.append("## Class weights from train split")
for class_id, class_name in ID2LABEL.items():
    summary_lines.append(f"- {class_name} ({class_id}): {class_weights[class_id].item():.4f}")
summary_lines.append("")
summary_lines.append("## Best validation checkpoint")
summary_lines.append(f"- Epoch: {best_epoch}")
summary_lines.append(f"- Validation Accuracy: {best_valid_metrics['accuracy']:.4f}")
summary_lines.append(f"- Validation Macro-F1: {best_valid_metrics['macro_f1']:.4f}")
summary_lines.append("")
summary_lines.append("## Final weighted BERT test result")
summary_lines.append(f"- Accuracy: {test_metrics['accuracy']:.4f}")
summary_lines.append(f"- Macro-F1: {test_metrics['macro_f1']:.4f}")
summary_lines.append("")
summary_lines.append("## Training history")
summary_lines.append("")
summary_lines.append(dataframe_to_markdown(history_export))
summary_lines.append("")
summary_lines.append("## Confusion matrix")
summary_lines.append("- Label order: [0, 1] = [REAL, FAKE]")
summary_lines.append("```")
summary_lines.append(str(test_metrics["confusion_matrix"]))
summary_lines.append("```")
summary_lines.append("")
summary_lines.append("## Classification report")
summary_lines.append("```")
summary_lines.append(test_metrics["classification_report"])
summary_lines.append("```")

out_dir = Path("results")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "liar_bert_weighted_baseline_run_output.md"
out_path.write_text("\n".join(summary_lines), encoding="utf-8")

print("Saved:", out_path.resolve())
